# **Kết nối Drive**

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Setup**

## **Tải thư viện (nếu chưa có)**

In [2]:
!pip -q install transformers datasets seqeval accelerate underthesea

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/43.6 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.4/978.4 kB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 54.5 MB/s eta 0:00:00


## **Import thư viện & cấu hình**

In [3]:
import os
import json
import random
import unicodedata
from typing import List, Dict, Tuple, Any, Optional
import json, math

import pandas as pd
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (AutoTokenizer,
                          AutoModelForTokenClassification,
                          TrainingArguments,
                          Trainer,
                          set_seed,)

from seqeval.metrics import f1_score, precision_score, recall_score
from underthesea import word_tokenize
from datasets import Features, Value, Sequence
from transformers import DataCollatorForTokenClassification
import torch
from transformers import Trainer
import torch.nn.functional as F
from seqeval.metrics import  classification_report


## **Các hàm tiện ích**

### Tiện ích xử lý JSONL & Unicode

- **`nfc(text)`**: Chuẩn hoá chuỗi văn bản về dạng Unicode NFC (quan trọng để tránh lỗi so khớp tiếng Việt).
- **`load_jsonl(path)`**: Đọc file `.jsonl`, chuyển đổi từng dòng thành dictionary; báo lỗi chi tiết nếu dòng sai định dạng.
- **`save_jsonl(rows, path)`**: Ghi danh sách dữ liệu ra file `.jsonl`, giữ nguyên hiển thị tiếng Việt (`ensure_ascii=False`).
- **`split_jsonl(...)`**: Trộn ngẫu nhiên và chia dữ liệu thành 3 tập **train/valid/test** theo tỷ lệ và seed cố định.
- **`read_jsonl_and_rename_label(path)`**: Đọc `.jsonl`, chuẩn hóa NFC và đổi khóa `label` → `labels` để đồng bộ schema đầu vào.

In [ ]:
def nfc(text: str) -> str:
    return unicodedata.normalize("NFC", text)

def load_jsonl(path: str) -> List[Dict[str, Any]]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip() 
            if not line: 
                continue
            try:
                rows.append(json.loads(line)) 
            except Exception as e:
                raise ValueError(f"Invalid JSON at {path}:{ln}\nLine: {line}\nError: {e}") 
    return rows

def save_jsonl(rows: List[Dict[str, Any]], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True) 
    with open(path, "w", encoding="utf-8") as f: 
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n") 

def split_jsonl(
    input_path: str,
    out_train: str,
    out_valid: str,
    out_test: str,
    train_ratio: float = 0.8,
    valid_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,) -> None:

    assert abs(train_ratio + valid_ratio + test_ratio - 1.0) < 1e-9
    rows = load_jsonl(input_path)
    rng = random.Random(seed)
    rng.shuffle(rows)

    n = len(rows)
    n_train = int(n * train_ratio)
    n_valid = int(n * valid_ratio)

    train_rows = rows[:n_train]
    valid_rows = rows[n_train:n_train + n_valid]
    test_rows  = rows[n_train + n_valid:]

    save_jsonl(train_rows, out_train)
    save_jsonl(valid_rows, out_valid)
    save_jsonl(test_rows, out_test)
    print(f"Split: train={len(train_rows)}, valid={len(valid_rows)}, test={len(test_rows)}")

def read_jsonl_and_rename_label(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                r = json.loads(line)
            except Exception as e:
                raise ValueError(f"Invalid JSON at {path}:{ln}\nLine: {line}\nError: {e}")

            # Đổi tên key từ 'label' sang 'labels' nếu cần để thống nhất đầu vào cho mô hình.
            if "labels" not in r and "label" in r:
                r["labels"] = r["label"]

            # Áp dụng chuẩn hóa NFC cho chuỗi văn bản.
            if "text" in r:
                r["text"] = nfc(r["text"])

            rows.append(r)
    return rows

### Kiểm tra & Làm sạch dữ liệu gán nhãn (Span-based)

- **`_check_no_overlap(spans)`**: Sắp xếp các span theo vị trí và phát hiện sự chồng lấn; báo lỗi ngay nếu tìm thấy overlap (không cho phép nested entity).
- **`validate_rows(rows, ...):`**
  - Kiểm tra bắt buộc trường `text` (chuẩn hoá NFC) và labels (phải là list).
  - Xác thực cấu trúc nhãn `[start, end, prob_dict]`:
    - **Offsets**: Phải là số nguyên dương, `start < end`, không vượt quá độ dài văn bản.
    - **Nội dung**: Không chấp nhận span rỗng hoặc chỉ chứa khoảng trắng (nếu `strict=True`).
    - **Xác suất (`prob_dict`)**: Giá trị không âm, tổng xác suất phải bằng 1 (có cơ chế tự chuẩn hóa).
  - Phát hiện **overlap span** trong cùng một câu.
  - Hỗ trợ loại bỏ dòng lỗi (`drop_invalid`) và ghi log chi tiết (`verbose`) để gỡ lỗi dữ liệu.
- **Kết quả**: Trả về danh sách các dòng dữ liệu sạch, đảm bảo đúng schema `{text, labels}`.

`_check_no_overlap`: Để đảm bảo không có hai thực thể nào đè lên nhau trong cùng một vị trí.

`validate_rows`: Kiểm tra xem các vị trí (offsets) của thực thể có nằm ngoài độ dài văn bản không, có bị trống không.

In [ ]:
def _check_no_overlap(spans):
    spans_sorted = sorted(spans, key=lambda x: (x[0], x[1]))
    for i in range(1, len(spans_sorted)):
        ps, pe = spans_sorted[i-1]
        cs, ce = spans_sorted[i]
        if cs < pe:
            raise ValueError(f"Overlapping spans detected: prev={spans_sorted[i-1]} cur={spans_sorted[i]}")

def validate_rows(rows, strict=True, drop_invalid=False, verbose=True):
    out = []
    removed = 0

    for idx, r in enumerate(rows):
        try:
            if "text" not in r:
                raise ValueError(f"Row {idx} missing 'text'. Row={r}")

            text = nfc(r["text"])
            labels = r.get("labels", []) or []

            if not isinstance(labels, list):
                raise ValueError(f"Row {idx} 'labels' must be a list, got {type(labels)}")

            spans_for_overlap = []
            new_labels = []

            for item in labels:
                if not (isinstance(item, (list, tuple)) and len(item) == 3):
                    raise ValueError(f"Row {idx} invalid label item: {item}")

                s, e, prob_dict = item
                if not (isinstance(s, int) and isinstance(e, int)):
                    raise ValueError(f"Row {idx} offsets must be int: {item}")
                if s < 0 or e < 0 or s > e:
                    raise ValueError(f"Row {idx} offsets invalid: {item}")

                if e > len(text):
                    msg = f"Row {idx} end out of range: {item}, len(text)={len(text)}"
                    if strict:
                        raise ValueError(msg)
                    e = min(e, len(text))

                if strict and (e - s) == 0:
                    raise ValueError(f"Row {idx} empty span: {item}")
                if strict and text[s:e].strip() == "":
                    raise ValueError(f"Row {idx} span is only whitespace: {item} -> '{text[s:e]}'")

                if not isinstance(prob_dict, dict) or len(prob_dict) == 0:
                    raise ValueError(f"Row {idx} prob_dict must be non-empty dict: {prob_dict}")

                clean = {}
                for k, v in prob_dict.items():
                    if not isinstance(k, str) or not k.strip():
                        raise ValueError(f"Row {idx} invalid type key: {k}")
                    fv = float(v)
                    if fv < 0:
                        raise ValueError(f"Row {idx} negative prob: {k}={fv}")
                    clean[k] = fv
                z = sum(clean.values())
                if z <= 0:
                    raise ValueError(f"Row {idx} sum(prob)=0: {clean}")
                if strict and abs(z - 1.0) > 1e-3:
                    raise ValueError(f"Row {idx} sum(prob) must be ~1, got {z}: {clean}")
                clean = {k: v / z for k, v in clean.items()}

                spans_for_overlap.append((s, e))
                new_labels.append([s, e, clean])

            _check_no_overlap(spans_for_overlap)

            out.append({"text": text, "labels": new_labels})

        except Exception as err:
            if strict and not drop_invalid:
                raise
            removed += 1
            if verbose:
                print(f"[DROP] Row {idx}: {err}")

    if verbose:
        print(f"Validation done | total={len(rows)} | kept={len(out)} | removed={removed}")

    return out

### Xây dựng hệ nhãn BIO cho NER (token-level)

- `build_bio_label_maps(rows_all)`:
  Duyệt toàn bộ dữ liệu để thu thập danh sách các **loại thực thể** (entity types) duy nhất xuất hiện.

- **Cách hoạt động**:
  - Trích xuất tên thực thể từ các khóa (keys) trong **dictionary xác suất** của mỗi nhãn (vị trí item[2]).
  - Xây dựng hệ nhãn **BIO** chuẩn bao gồm:
    - `O`: Token không thuộc thực thể.
    - `B-<ENTITY>`: Token bắt đầu thực thể.
    - `I-<ENTITY>`: Token nằm bên trong thực thể.
  - Sắp xếp danh sách thực thể trước khi tạo ID để đảm bảo **thứ tự index cố định** giữa các lần chạy.

- **Output**:
  - `labels`: Danh sách toàn bộ nhãn BIO (List[str]).
  - `label2id`: Ánh xạ nhãn → số nguyên (dùng khi train).
  - `id2label`: Ánh xạ số nguyên → nhãn (dùng khi inference).

- **Mục đích**:
  Chuẩn hoá không gian nhãn đầu ra, chuyển đổi dữ liệu gán nhãn thô sang định dạng số hoá phục vụ huấn luyện mô hình NER.

In [ ]:
def build_bio_label_maps(rows_all):
    entity_types = set()
    for r in rows_all:
        for item in (r.get("labels", []) or []):
            if isinstance(item, (list, tuple)) and len(item) == 3 and isinstance(item[2], dict):
                for typ in item[2].keys():
                    entity_types.add(str(typ))

    labels = ["O"]
    # Với mỗi loại thực thể, tự động tạo các nhãn B- (bắt đầu thực thể) và I- (bên trong thực thể).
    for ent in sorted(entity_types):
        labels.append(f"B-{ent}")
        labels.append(f"I-{ent}")

    label2id = {l: i for i, l in enumerate(labels)}
    id2label = {i: l for l, i in label2id.items()}
    return labels, label2id, id2label

### Chuyển dữ liệu Span-level sang HuggingFace Dataset

- `FEATURES`: Định nghĩa schema chuẩn cho HuggingFace Dataset, bao gồm:
  - `text`: Văn bản gốc (đã chuẩn hóa NFC).
  - `spans_start`, `spans_end`: Danh sách vị trí ký tự bắt đầu/kết thúc span.
  - `spans_label`: Nhãn có xác suất cao nhất (**Top-1** fallback).
  - `spans_prob`: Chuỗi JSON lưu phân phối xác suất đầy đủ (Soft Label).
  
- `to_dataset(rows)`:
  - Nhận dữ liệu đầu vào `{text, labels=[[start, end, prob_dict], ...]}`.
  - Tách thông tin span thành 4 mảng song song: vị trí, nhãn chính và prob_dict (serialized JSON).
  - Chuyển đổi danh sách record sang đối tượng **HuggingFace** `Dataset` tuân thủ cấu trúc `FEATURES`.

- **Mục đích**:
  Chuẩn hoá dữ liệu gán nhãn ở mức **character-span** kèm thông tin xác suất mềm, phục vụ việc căn chỉnh (align) sang token và huấn luyện mô hình với hàm mất mát phân phối (như KL Divergence).

In [ ]:
FEATURES = Features({
    "text": Value("string"),
    "spans_start": Sequence(Value("int32")),
    "spans_end": Sequence(Value("int32")),
    "spans_label": Sequence(Value("string")),  # top1 fallback
    "spans_prob": Sequence(Value("string")),   # json string of {TYPE: prob}
})

def to_dataset(rows):
    items = []
    for r in rows:
        text = nfc(r["text"])
        raw = r.get("labels", []) or []  # [[start, end, {label: prob}], ...]

        starts, ends, labs, probs = [], [], [], []
        for s, e, prob_dict in raw:
            s = int(s); e = int(e)
            prob_dict = prob_dict if isinstance(prob_dict, dict) else {}
            
            top1 = max(prob_dict, key=prob_dict.get) if prob_dict else "O"

            starts.append(s)
            ends.append(e)
            labs.append(str(top1))
            probs.append(json.dumps(prob_dict, ensure_ascii=False))

        items.append({
            "text": text,
            "spans_start": starts,
            "spans_end": ends,
            "spans_label": labs,
            "spans_prob": probs,
        })
        
    # Tạo đối tượng Dataset cuối cùng với các FEATURES đã định nghĩa.
    return Dataset.from_list(items, features=FEATURES)

### Các hàm Căn chỉnh & Chuyển đổi nhãn (Label Alignment)

- `build_char_mask(...)`: Tạo bản đồ ánh xạ từng ký tự trong văn bản sang nhãn thực thể, đóng vai trò trung gian để giải quyết các trường hợp chồng lấn (overlap).

- **Nhánh 1: Fast Tokenizer (`align_spans_to_token_soft_labels`)**:
  - Dành cho các mô hình hỗ trợ `offset_mapping` (như DeBERTa V3).
  - Ánh xạ trực tiếp tọa độ ký tự sang token để tạo **Soft Labels**, giúp mô hình học được độ bất định của dữ liệu.

- **Nhánh 2: Robust Alignment (`align_spans_to_token_labels_phobert`)**:
  - Dành cho các mô hình cần tách từ trước (như PhoBERT).
  - Sử dụng cơ chế **Pointer Scanning** (`_scan_match_token`) để khớp chính xác vị trí từ trong văn bản gốc.
  - Thực hiện quy trình 3 bước: Span ký tự $\to$ Nhãn từ (Word-level) $\to$ Nhãn token con (Subword-level).

- **Tiện ích hỗ trợ**:
  - `sanity_check_alignment`: Kiểm tra tính toàn vẹn dữ liệu (kích thước input khớp với nhãn) trước khi đưa vào huấn luyện.
  - `_as_prob_dict`: Chuẩn hóa dữ liệu đầu vào (JSON/Dict) về định dạng từ điển xác suất thống nhất.

In [ ]:
def nfc(s: str) -> str:
    return unicodedata.normalize("NFC", s or "")

# ---------------------------
# Helper: build char mask (hard label per char)
# ---------------------------
def build_char_mask(text: str, spans: List[Tuple[int, int, Any]], *, prefer="first"):
    text = nfc(text)
    mask = [None] * len(text)

    # Sort span theo start, nếu cùng start thì span dài hơn ưu tiên
    spans_sorted = sorted(
        [(int(s), int(e), lab) for s, e, lab in spans],
        key=lambda x: (x[0], -(x[1]-x[0]))
    )

    def _to_hard_label(lab_any: Any) -> Optional[str]:
        if isinstance(lab_any, str):
            return lab_any
        if isinstance(lab_any, dict) and len(lab_any) > 0:
            # top1 label for mask purpose
            k = max(lab_any, key=lambda x: float(lab_any.get(x, 0.0)))
            return str(k)
        return None

    for s, e, lab_any in spans_sorted:
        s = max(0, min(s, len(text)))
        e = max(0, min(e, len(text)))
        if e <= s:
            continue

        lab = _to_hard_label(lab_any)
        if not lab:
            continue

        for i in range(s, e):
            if prefer == "first":
                if mask[i] is None:
                    mask[i] = lab
            else:
                mask[i] = lab

    return mask


# ==========================================================
# A) FAST TOKENIZER WITH OFFSETS (soft labels)
# ==========================================================


def _as_prob_dict(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            return json.loads(x)
        except Exception:
            return {}
    return {}

def align_spans_to_token_soft_labels(tokenizer, label2id, examples):
    texts = examples["text"]
    starts_batch = examples["spans_start"]
    ends_batch   = examples["spans_end"]

    # spans_prob có thể không có (fallback hard-only), hoặc có nhưng là JSON string
    probs_batch  = examples.get("spans_prob", None)

    enc = tokenizer(texts, truncation=True, return_offsets_mapping=True)
    offsets_batch = enc["offset_mapping"]

    num_labels = len(label2id)
    o_id = label2id["O"]

    all_hard, all_soft = [], []

    for idx, (offsets, starts, ends) in enumerate(zip(offsets_batch, starts_batch, ends_batch)):
        hard = []
        soft = []

        # init O=1
        for (os, oe) in offsets:
            if os == 0 and oe == 0:  # special token
                hard.append(-100)
                soft.append([0.0]*num_labels)
            else:
                hard.append(o_id)
                v = [0.0]*num_labels
                v[o_id] = 1.0
                soft.append(v)

        if probs_batch is None:
            # fallback hard-only từ spans_label nếu bạn muốn (tuỳ bạn)
            # ở đây: giữ nguyên O nếu không có soft
            pass
        else:
            span_probs_list = probs_batch[idx] or []
            for s, e, prob_x in zip(starts, ends, span_probs_list):
                s = int(s); e = int(e)
                prob_dict = _as_prob_dict(prob_x)

                first = True
                for i, (os, oe) in enumerate(offsets):
                    if os == 0 and oe == 0:
                        continue
                    if oe <= s or os >= e:
                        continue

                    vec = [0.0]*num_labels
                    for typ, p in prob_dict.items():
                        if p is None:
                            continue
                        tag = ("B-" if first else "I-") + str(typ)
                        if tag in label2id:
                            vec[label2id[tag]] += float(p)

                    if sum(vec) <= 0:
                        vec[o_id] = 1.0
                    else:
                        z = sum(vec)
                        vec = [x/z for x in vec]

                    soft[i] = vec
                    hard[i] = int(np.argmax(vec))
                    first = False

        all_hard.append(hard)
        all_soft.append(soft)

    enc.pop("offset_mapping")
    enc["labels"] = all_hard
    enc["soft_labels"] = all_soft
    # Kết quả trả về gồm cả nhãn cứng (labels) và phân phối xác suất (soft_labels).
    return enc



def build_bio_label_maps(rows_all):
    entity_types = set()
    for r in rows_all:
        current_labels = r.get("labels", []) or []
        for item in current_labels:
            if isinstance(item, (list, tuple)) and len(item) == 3 and isinstance(item[2], dict):
                for typ in item[2].keys():
                    entity_types.add(str(typ))

    labels = ["O"]
    for ent in sorted(entity_types):
        labels.append(f"B-{ent}")
        labels.append(f"I-{ent}")

    label2id = {l: i for i, l in enumerate(labels)} 
    id2label = {i: l for l, i in label2id.items()}
    
    return labels, label2id, id2label


def _scan_match_token(text: str, token: str, pos: int):
    """
    Tìm vị trí khớp của một token đã phân đoạn vào văn bản gốc bằng cơ chế con trỏ (pointer scanning).
    Mục đích:
    - Ánh xạ từ token (có thể chứa '_' thay cho khoảng trắng) về tọa độ (start, end) chính xác trong text.
    - Bỏ qua các khoảng trắng thừa và xử lý sai lệch định dạng giữa bộ tách từ và văn bản gốc.

    Args:
        text (str): Văn bản gốc (đã chuẩn hóa NFC).
        token (str): Token cần khớp (ví dụ: "bệnh_nhân").
        pos (int): Vị trí bắt đầu quét trong văn bản gốc.
    Returns:
        Tuple[Optional[int], Optional[int], int]: (start_index, end_index, next_pos). 
        Trả về (None, None, pos) nếu không tìm thấy khớp.
    """
    
    raw = token.replace("_", " ")
    n = len(text)

    while pos < n and text[pos].isspace():
        pos += 1
    if not raw:
        return None, None, pos

    first = raw[0]
    scan = pos
    while scan < n:
        if text[scan] != first:
            scan += 1
            continue
        end = scan + len(raw)
        if end <= n and text[scan:end] == raw:
            return scan, end, end
        scan += 1

    return None, None, pos


def viet_words_with_offsets_robust(text: str):
    """
    Tách từ tiếng Việt và xác định tọa độ của từng từ trong văn bản gốc.
    Mục đích:
    - Kết hợp bộ tách từ (như underthesea) với hàm scan khớp vị trí để tạo ra danh sách token kèm offset.
    - Đảm bảo tính nhất quán giữa văn bản thô và văn bản sau khi phân đoạn từ.

    Args:
        text (str): Văn bản tiếng Việt cần xử lý.
    Returns:
        List[Tuple[str, int, int]]: Danh sách các bộ (token, start_offset, end_offset).
    """
    text = nfc(text)
    tokens = _tokenize_vi_words(text)

    out = []
    pos = 0
    for tok in tokens:
        s, e, pos2 = _scan_match_token(text, tok, pos)
        if s is None:
            continue
        out.append((tok, s, e))
        pos = pos2
    return out


def _make_word_soft_tags_robust(
    text: str,
    spans_start: List[int],
    spans_end: List[int],
    spans_prob: List[Dict[str, float]],
    label2id: Dict[str, int],
):
    """ 
    Chuyển đổi các thực thể mức ký tự (soft spans) sang nhãn mức từ (word-level).
    Mục đích:
    - Tính toán phân phối xác suất nhãn (soft labels) cho từng từ dựa trên mức độ chồng lấn với thực thể.
    - Áp dụng quy tắc BIO: từ đầu tiên trong thực thể nhận nhãn B-*, các từ tiếp theo nhận nhãn I-*.
    - Hỗ trợ học từ nhãn mềm (nhiều xác suất nhãn trên cùng một đơn vị từ).

    Args:
        text (str): Văn bản gốc.
        spans_start (List[int]): Danh sách vị trí bắt đầu của các thực thể.
        spans_end (List[int]): Danh sách vị trí kết thúc của các thực thể.
        spans_prob (List[Dict[str, float]]): Danh sách các từ điển xác suất nhãn tương ứng.
        label2id (Dict[str, int]): Bảng ánh xạ từ tên nhãn sang ID số.
    Returns:
        Tuple[List, List[str], List[List[float]]]: 
            - words_info: Danh sách (token, start, end).
            - word_hard: Danh sách nhãn cứng (BIO tags) cho từng từ.
            - word_soft: Ma trận xác suất nhãn mềm cho từng từ.
    """
    
    text = nfc(text)
    num_labels = len(label2id)
    o_id = label2id["O"]

    words_info = viet_words_with_offsets_robust(text)  # [(tok, ws, we)]
    word_soft = [[0.0] * num_labels for _ in words_info]
    word_hard = ["O"] * len(words_info)

    # init O=1
    for i in range(len(words_info)):
        word_soft[i][o_id] = 1.0
    # apply each span to words that overlap it
    for s, e, prob_dict in zip(spans_start, spans_end, spans_prob):
        s = int(s); e = int(e)
        first_word = True

        for wi, (_, ws, we) in enumerate(words_info):
            if we <= s or ws >= e:
                continue

            prob_dict = _as_prob_dict(prob_dict)

            vec = [0.0] * num_labels
            for typ, p in (prob_dict or {}).items():
                tag = ("B-" if first_word else "I-") + str(typ)
                if tag in label2id:
                    vec[label2id[tag]] += float(p)

            if sum(vec) <= 0:
                vec = [0.0] * num_labels
                vec[o_id] = 1.0
            else:
                z = sum(vec)
                vec = [x / z for x in vec]

            word_soft[wi] = vec
            word_hard[wi] = max(label2id.keys(), key=lambda k: vec[label2id[k]])  # argmax label name
            first_word = False

    return words_info, word_hard, word_soft


def encode_words_and_labels(
    tokenizer,
    tokens: List[str],
    word_tags: List[str],
    label2id: Dict[str, int],
    max_length=512,
    word_soft: Optional[List[List[float]]] = None,   
):
    """
    Tokenize văn bản theo từng từ đã phân đoạn và căn chỉnh nhãn mức token (subword-level).
    Mục đích:
    - Chuyển đổi từ mức từ (word-level) sang mức token thực tế của mô hình (ví dụ: PhoBERT BPE).
    - Xử lý logic nhãn BIO cho subwords: Chỉ token đầu tiên của một từ giữ nhãn gốc (B- hoặc I-), 
      các token con phía sau của từ đó luôn được chuyển về nhãn I- tương ứng.
    - Căn chỉnh nhãn mềm (soft labels): Truyền phân phối xác suất từ mức từ xuống mức token, 
      đồng thời tự động chuyển dịch trọng số từ B- sang I- cho các subwords phía sau.

    Args:
        tokenizer: Bộ giải mã ngôn ngữ (Tokenizer) của mô hình.
        tokens (List[str]): Danh sách các từ đã phân đoạn (ví dụ: ["bệnh_nhân", "đau", "đầu"]).
        word_tags (List[str]): Danh sách các nhãn BIO tương ứng với từng từ.
        label2id (Dict[str, int]): Bảng ánh xạ nhãn sang ID.
        max_length (int): Độ dài tối đa của chuỗi token đầu ra.
        word_soft (Optional[List[List[float]]]): Ma trận xác suất nhãn mềm mức từ.

    Returns:
        Dict[str, Any]: Dictionary chứa input_ids, attention_mask, labels và soft_labels.
    """
    
    num_labels = len(label2id)
    o_id = label2id["O"]

    input_ids = []
    labels = []
    soft_labels = [] if word_soft is not None else None

    for idx, (tok, tag) in enumerate(zip(tokens, word_tags)):
        ids = tokenizer.encode(tok, add_special_tokens=False)
        if not ids:
            continue

        # base vec for this word
        if word_soft is not None:
            base_vec = word_soft[idx]
            # clamp/normalize tiny numeric drift
            z = float(sum(base_vec))
            if z <= 0:
                base_vec = [0.0] * num_labels
                base_vec[o_id] = 1.0
            else:
                base_vec = [float(x) / z for x in base_vec]

        if tag == "O":
            input_ids.extend(ids)
            labels.extend([o_id] * len(ids))
            if soft_labels is not None:
                soft_labels.extend([base_vec] * len(ids))
        else:
            # first subword keeps tag
            input_ids.append(ids[0])
            labels.append(label2id[tag])

            if soft_labels is not None:
                # first subword uses base_vec as-is
                soft_labels.append(base_vec)

            # later subwords: convert B- -> I-
            itag = ("I-" + tag[2:]) if tag.startswith("B-") else tag
            input_ids.extend(ids[1:])
            labels.extend([label2id[itag]] * (len(ids) - 1))

            if soft_labels is not None and len(ids) > 1:
                # move distribution mass from B-* to I-* for later pieces
                vec2 = base_vec
                if tag.startswith("B-"):
                    ent = tag[2:]
                    b = f"B-{ent}"
                    i = f"I-{ent}"
                    if b in label2id and i in label2id:
                        vec2 = base_vec.copy()
                        vec2[label2id[i]] += vec2[label2id[b]]
                        vec2[label2id[b]] = 0.0
                        z2 = sum(vec2)
                        vec2 = [x / z2 for x in vec2] if z2 > 0 else vec2
                soft_labels.extend([vec2] * (len(ids) - 1))

        if len(input_ids) >= (max_length - 2):
            input_ids = input_ids[: (max_length - 2)]
            labels = labels[: (max_length - 2)]
            if soft_labels is not None:
                soft_labels = soft_labels[: (max_length - 2)]
            break

    input_ids = tokenizer.build_inputs_with_special_tokens(input_ids)
    labels = [-100] + labels + [-100]
    attention_mask = [1] * len(input_ids)

    out = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

    if soft_labels is not None:
        soft_labels = [[0.0] * num_labels] + soft_labels + [[0.0] * num_labels]
        out["soft_labels"] = soft_labels

    return out


def align_spans_to_token_labels_phobert(tokenizer, label2id: Dict[str,int], examples: Dict[str,Any], max_length=512):
    """
    Robust PhoBERT alignment:
    - sử dụng spans_start/spans_end/spans_prob để tạo labels + soft_labels
    - fallback chỉ sử dụng spans_label (top1) nếu spans_prob bị thiếu
    
    Args:
        tokenizer: PhoBERT tokenizer.
        label2id (Dict[str, int]): Mapping label -> id.
        examples (Dict): Batch examples.
        max_length (int): Độ dài max (default 512).
    
    Returns:
        Dict: Encoding với input_ids, attention_mask, labels, soft_labels.
    """

    out = {"input_ids": [], "attention_mask": [], "labels": [], "soft_labels": []}

    # required keys
    texts = examples["text"]
    starts_batch = examples["spans_start"]
    ends_batch   = examples["spans_end"]

    probs_batch = examples.get("spans_prob", None)

    for idx, (text, starts, ends) in enumerate(zip(texts, starts_batch, ends_batch)):
        text = nfc(text)

        if probs_batch is None:
            # fallback hard-only from spans_label (old behavior)
            labs = examples["spans_label"][idx]
            spans_hard = [(int(s), int(e), lab) for s, e, lab in zip(starts, ends, labs)]
            # build word tags hard
            words_info = viet_words_with_offsets_robust(text)
            tokens = [w for (w, _, _) in words_info]

            # word-level BIO tags via char mask
            mask = build_char_mask(text, spans_hard, prefer="first")
            begin_set = {(max(0, min(int(s), len(text))), lab) for (s, e, lab) in spans_hard}

            word_tags = []
            for (_, ws, we) in words_info:
                lab = None
                for i in range(ws, min(we, len(mask))):
                    if mask[i] is not None:
                        lab = mask[i]
                        break
                if lab is None:
                    word_tags.append("O")
                else:
                    word_tags.append(f"B-{lab}" if ((ws, lab) in begin_set) else f"I-{lab}")

            enc = encode_words_and_labels(tokenizer, tokens, word_tags, label2id, max_length=max_length)
            out["input_ids"].append(enc["input_ids"])
            out["attention_mask"].append(enc["attention_mask"])
            out["labels"].append(enc["labels"])
            # soft_labels: make one-hot from labels for compatibility
            num_labels = len(label2id)
            soft = []
            for lab_id in enc["labels"]:
                if lab_id == -100:
                    soft.append([0.0]*num_labels)
                else:
                    v = [0.0]*num_labels
                    v[int(lab_id)] = 1.0
                    soft.append(v)
            out["soft_labels"].append(soft)
            continue

        # NEW soft path
        span_probs = probs_batch[idx]
        words_info, word_tags_hard, word_soft = _make_word_soft_tags_robust(
            text, starts, ends, span_probs, label2id
        )
        tokens = [w for (w, _, _) in words_info]

        enc = encode_words_and_labels(
            tokenizer,
            tokens,
            word_tags_hard,
            label2id,
            max_length=max_length,
            word_soft=word_soft,
        )

        out["input_ids"].append(enc["input_ids"])
        out["attention_mask"].append(enc["attention_mask"])
        out["labels"].append(enc["labels"])
        out["soft_labels"].append(enc["soft_labels"])

    return out


# ==========================================================
# Sanity check
# ==========================================================
def sanity_check_alignment(batch_enc: Dict[str,Any], n=5):
    """
    Kiểm tra tính nhất quán về độ dài giữa các thành phần dữ liệu sau khi căn chỉnh (alignment).
    Mục đích:
    - Đảm bảo tính toàn vẹn kỹ thuật: Trong mô hình Transformer, mỗi token đầu vào (input_ids) 
      phải đi kèm chính xác với một nhãn (hard label) và một vector xác suất (soft label).
    - Ngăn ngừa lỗi "RuntimeError" khi huấn luyện do lệch kích thước ma trận (dimension mismatch).
    - Xác nhận rằng các special tokens ([CLS], [SEP], [PAD]) đã được xử lý đúng độ dài.

    Args:
        batch_enc (Dict[str, Any]): Dictionary chứa dữ liệu đã được tokenize và căn chỉnh nhãn 
                                     (thường gồm: input_ids, labels, attention_mask, soft_labels).
        n(int): Số lượng mẫu đầu tiên trong batch cần thực hiện kiểm tra (mặc định là 5 mẫu).
    Returns:
        bool: Trả về True nếu tất cả các mẫu được kiểm tra đều vượt qua assert.
    Raises:
        AssertionError: Nếu phát hiện sự sai lệch về độ dài giữa IDs, Labels hoặc Mask.
    """
    
    for i in range(min(n, len(batch_enc["input_ids"]))):
        a = len(batch_enc["input_ids"][i])
        b = len(batch_enc["labels"][i])
        c = len(batch_enc["attention_mask"][i]) if "attention_mask" in batch_enc else a
        
        # Kiểm tra sự khớp nhau giữa ID, nhãn và mask
        assert a == b == c, f"Length mismatch at sample {i}: ids={a}, labels={b}, mask={c}"

        if "soft_labels" in batch_enc:
            d = len(batch_enc["soft_labels"][i])
            assert a == d, f"Soft length mismatch at sample {i}: ids={a}, soft={d}"
    return True

def has_offset_mapping(tokenizer) -> bool:
    """
    Kiểm tra xem Tokenizer đang dùng có phải là loại 'Fast' (hỗ trợ ánh xạ vị trí - offset mapping) hay không.
    """
    return bool(getattr(tokenizer, "is_fast", False))

def _as_prob_dict(x):
    """
    Hàm tiện ích giúp chuẩn hóa dữ liệu xác suất thực thể về định dạng từ điển (dictionary).
    Mục đích:
    - Xử lý linh hoạt các loại dữ liệu đầu vào khác nhau (None, Dictionary, hoặc chuỗi JSON string) thường gặp khi đọc từ file .jsonl.
    - Đảm bảo tính ổn định cho quy trình huấn luyện bằng cách luôn trả về một đối tượng dict (có thể rỗng) thay vì gây lỗi chương trình.

    Args:
        x (Any): Dữ liệu xác suất đầu vào của thực thể.
    Returns:
        Dict[str, float]: chứa thông tin nhãn và xác suất tương ứng. Trả về {} nếu dữ liệu không hợp lệ hoặc rỗng.
    """
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            return json.loads(x)
        except Exception:
            return {}
    return {}

### Hàm đánh giá cho NER (BIO tagging)

- `build_metrics_fn(id2label)`: Khởi tạo hàm `compute_metrics` chuyên dụng để tích hợp vào HuggingFace `Trainer`.

- **Cách hoạt động**:
  - Nhận **logits** và **labels** từ mô hình, xác định nhãn dự đoán bằng hàm `argmax`.
  - **Lọc bỏ token đặc biệt** (`label = -100`) để đảm bảo chỉ đánh giá trên các từ có nghĩa (bỏ qua padding/subwords).
  - Chuyển đổi **ID số** $\to$ **nhãn BIO** thực tế thông qua từ điển ánh xạ `id2label`.
  - Tính toán **Precision, Recall, F1-score** ở mức thực thể (entity-level) bằng thư viện chuẩn `seqeval`.

- **Kết quả**: Trả về dictionary chứa các chỉ số hiệu năng, phục vụ việc theo dõi và so sánh chất lượng mô hình trong quá trình huấn luyện.

In [ ]:
def build_metrics_fn(id2label: Dict[int, str]):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        true_preds, true_labels = [], []
        for p, y in zip(preds, labels):
            p_seq, y_seq = [], []
            for pi, yi in zip(p, y):
                if yi == -100:
                    continue
                p_seq.append(id2label[int(pi)])
                y_seq.append(id2label[int(yi)])
            true_preds.append(p_seq)
            true_labels.append(y_seq)

        return {
            "precision": precision_score(true_labels, true_preds),
            "recall": recall_score(true_labels, true_preds),
            "f1": f1_score(true_labels, true_preds),
        }
    return compute_metrics

# **Data preparation**

### Chia dữ liệu JSONL thành train / valid / test

- `split_jsonl(...)`: Đọc dữ liệu nguồn từ Drive (`data.jsonl`), trộn ngẫu nhiên và phân tách thành 3 tập dữ liệu theo tỷ lệ:
  - **Train**: 80% (dùng để huấn luyện).
  - **Validation**: 10% (dùng để tinh chỉnh tham số).
  - **Test**: 10% (dùng để đánh giá độc lập).
- `seed=1`: Thiết lập hạt giống ngẫu nhiên để đảm bảo kết quả phân chia có thể tái lập (reproducible) chính xác giữa các lần chạy.
- **Output**: Lưu kết quả vào thư mục cục bộ `data/` gồm: `train.jsonl`, `valid.jsonl`, `test.jsonl`.

In [ ]:
split_jsonl(
    input_path="/content/drive/MyDrive/medical_ner/data/data_new_method.jsonl",
    out_train="data/train.jsonl",
    out_valid="data/valid.jsonl",
    out_test="data/test.jsonl",
    train_ratio=0.8,
    valid_ratio=0.1,
    test_ratio=0.1,
    seed=1,
)

Split: train=2411, valid=301, test=302


### Chuẩn bị dữ liệu & DatasetDict cho huấn luyện NER

- `set_seed(42)`: Cố định hạt giống ngẫu nhiên để đảm bảo tính tái lập cho toàn bộ quy trình.
- **Đọc & Kiểm tra dữ liệu**:
  - `read_jsonl_and_rename_label`: Đọc file và chuẩn hoá khóa dữ liệu (`label` $\to$ `labels`).
  - `validate_rows(..., strict=True, drop_invalid=True)`: Kiểm tra chặt chẽ cấu trúc span, Unicode NFC, logic xác suất; tự động **loại bỏ các mẫu lỗi** và ghi log chi tiết.

- **Xây dựng hệ nhãn BIO**:
  - `build_bio_label_maps(train + valid + test)`: Quét toàn bộ dữ liệu để tạo không gian nhãn `LABELS` và các từ điển ánh xạ (`label2id`, `id2label`) thống nhất.

- **Tạo HuggingFace** `DatasetDict`:
  - Chuyển đổi dữ liệu sang định dạng `Dataset` chuẩn (bao gồm cả `spans_prob` cho nhãn mềm) thông qua hàm `to_dataset`.
  - Tổ chức thành cấu trúc `DatasetDict` với các phân vùng: `train`, `validation`, `test`.

**Kết quả**: Bộ dữ liệu sạch, hệ nhãn BIO nhất quán, sẵn sàng cho bước Tokenization.

In [12]:
set_seed(42)

train_rows = validate_rows(read_jsonl_and_rename_label("data/train.jsonl"), strict=True, drop_invalid=True, verbose=True)
valid_rows = validate_rows(read_jsonl_and_rename_label("data/valid.jsonl"), strict=True, drop_invalid=True, verbose=True)
test_rows  = validate_rows(read_jsonl_and_rename_label("data/test.jsonl"), strict=True, drop_invalid=True, verbose=True)

print(f"Loaded: train={len(train_rows)}, valid={len(valid_rows)}, test={len(test_rows)}")

LABELS, label2id, id2label = build_bio_label_maps(train_rows + valid_rows + test_rows)
print("BIO LABELS:", LABELS)

ds = DatasetDict({
    "train": to_dataset(train_rows),
    "validation": to_dataset(valid_rows),
    "test": to_dataset(test_rows),
})
print(ds['test'][20])

Validation done | total=2411 | kept=2411 | removed=0
Validation done | total=301 | kept=301 | removed=0
[DROP] Row 122: Row 122 end out of range: [53, 61, {'PLANT_PART': 0.91018, 'HERB': 0.08982}], len(text)=53
Validation done | total=302 | kept=301 | removed=1
Loaded: train=2411, valid=301, test=301
BIO LABELS: ['O', 'B-DISEASE', 'I-DISEASE', 'B-DOSAGE', 'I-DOSAGE', 'B-HERB', 'I-HERB', 'B-HUMAN_PART', 'I-HUMAN_PART', 'B-PLANT_PART', 'I-PLANT_PART', 'B-SYMPTOM', 'I-SYMPTOM']
{'text': 'Lớp tế bào ngoài cùng của nội nhũ thường có màu sậm và chứa nhiều protid gọi là tầng chứa protid (lớp chứa alơron) góp phần quan trọng lúc hạt nảy mầm vì chứa nhiều phân hoá tố.', 'spans_start': [0, 26, 66, 80, 98, 139, 147], 'spans_end': [10, 33, 72, 96, 113, 142, 150], 'spans_label': ['PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART', 'PLANT_PART'], 'spans_prob': ['{"PLANT_PART": 0.89611, "HERB": 0.10389}', '{"PLANT_PART": 0.84257, "HERB": 0.15743}', '{"PLANT_PART": 0.92

### Bổ sung xác suất nhãn (Probability Imputation)

- `add_spans_prob_from_spans_label(examples)`:
  - Tự động rà soát và điền dữ liệu vào cột `spans_prob`.
  - **Cơ chế Fallback**:
    - Nếu đã có thông tin xác suất (Soft Label) $\to$ **Giữ nguyên**.
    - Nếu thiếu thông tin $\to$ Tự động tạo xác suất **100% (One-hot)** từ nhãn cứng (`spans_label`).

- `ds.map(...)`: Áp dụng hàm trên toàn bộ dataset (`batched=True`) để chuẩn hóa dữ liệu.

- **Mục đích**:
  - Đảm bảo 100% thực thể đều có trường phân phối xác suất hợp lệ.
  - Đồng bộ hóa định dạng dữ liệu đầu vào cho hàm mất mát **KL Divergence** (tránh lỗi khi tính toán loss trên nhãn cứng thuần túy).

In [ ]:

def add_spans_prob_from_spans_label(examples):
    # Kiểm tra xem trường spans_prob đã tồn tại trong dữ liệu hay chưa
    has_prob = "spans_prob" in examples and examples["spans_prob"] is not None

    out_probs = []
    # Duyệt qua từng dòng dữ liệu trong batch
    for i, labs in enumerate(examples["spans_label"]):
        labs = labs or []
        
        # Lấy danh sách xác suất hiện có (nếu có)
        if has_prob:
            cur = examples["spans_prob"][i] or []
        else:
            cur = []

        # cur có thể dài ngắn khác labs -> mình align theo labs
        new = []
        # Duyệt qua từng nhãn thực thể để căn chỉnh xác suất
        for j, lab in enumerate(labs):
            if j < len(cur) and cur[j]:
                # giữ nguyên soft prob hiện có (string json)
                new.append(cur[j])
            else:
                # thiếu thì mới tạo one-hot
                new.append(json.dumps({str(lab): 1.0}, ensure_ascii=False))
        out_probs.append(new)

    return {"spans_prob": out_probs}
ds = ds.map(add_spans_prob_from_spans_label, batched=True, desc="Fill spans_prob if missing")
ds

Fill spans_prob if missing:   0%|          | 0/2411 [00:00<?, ? examples/s]

Fill spans_prob if missing:   0%|          | 0/301 [00:00<?, ? examples/s]

Fill spans_prob if missing:   0%|          | 0/301 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob'],
        num_rows: 2411
    })
    validation: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob'],
        num_rows: 301
    })
    test: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob'],
        num_rows: 301
    })
})

### Kiểm tra & Đảm bảo tính nhất quán dữ liệu (Sanity Check)

- `_safe_json_load(x)`: Hàm tiện ích giúp giải mã JSON an toàn; xử lý linh hoạt các kiểu dữ liệu đầu vào (`str`, `dict`, `None`) để ngăn chặn lỗi runtime khi gặp dữ liệu nhiễu.

- `sanity_check_spans_prob(ds_split)`: Thực hiện rà soát logic toàn diện trên tập mẫu:
  - **Length Consistency**: Đảm bảo các trường thông tin thực thể (`start`, `end`, `label`, `prob`) có số lượng phần tử khớp nhau tuyệt đối trong mỗi bản ghi.
  - **Range Validity**: Xác thực tọa độ span (`start <= end`) phải nằm trong giới hạn độ dài văn bản gốc.
  - **Prob Format & Sum**: Kiểm tra định dạng JSON và đảm bảo tổng xác suất của Soft Label xấp xỉ 1.0.

- **Mục đích**: Phát hiện sớm các lỗi cấu trúc (structural errors) và sai lệch dữ liệu trước khi bước vào giai đoạn huấn luyện.

In [ ]:

def _safe_json_load(x):
    if x is None:
        return None
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return None
        return json.loads(x)
    return None

def sanity_check_spans_prob(ds_split, n=50):
    bad = 0
    for i in range(min(n, len(ds_split))):
        ex = ds_split[i]
        a,b,c,d = map(len, (ex["spans_start"], ex["spans_end"], ex["spans_label"], ex.get("spans_prob", [])))
        if not (a==b==c==d):
            print("[LEN_MISMATCH]", i, a,b,c,d)
            bad += 1
            continue

        text_len = len(ex["text"])
        for s,e,p in zip(ex["spans_start"], ex["spans_end"], ex.get("spans_prob", [])):
            if not (0 <= s <= e <= text_len):
                print("[RANGE_BAD]", i, (s,e), "len(text)=", text_len)
                bad += 1
                break
            pd = _safe_json_load(p)
            if pd is None:
                print("[PROB_PARSE_BAD]", i, p)
                bad += 1
                break
            sm = sum(v for v in pd.values() if v is not None)
            if sm > 0 and not (abs(sm - 1.0) < 1e-3):
                # không bắt buộc, nhưng nên gần 1
                pass
    print("bad =", bad)

sanity_check_spans_prob(ds["train"], n=50)

bad = 0


# **Training**

## Tokenize & Căn chỉnh nhãn (Span $\to$ Token)

- **Khởi tạo Model & Tokenizer:**
  - Lựa chọn mô hình huấn luyện: `microsoft/deberta-v3-large`.
  - Khởi tạo `AutoTokenizer` với `use_fast=True`.
  - `has_offset_mapping(...)`: Tự động phát hiện khả năng của tokenizer để chọn nhánh xử lý phù hợp.

- **Luồng xử lý Căn chỉnh (Alignment Logic)**:
  - **Nhánh 1 - Fast Tokenizer (DeBERTa/XLM-R)**:
    - Kích hoạt khi `use_offsets=True`.
    - Sử dụng `align_spans_to_token_soft_labels`: Tận dụng tọa độ ký tự để ánh xạ trực tiếp sang token và tạo **Soft Labels**.
  - **Nhánh 2 - Slow Tokenizer (PhoBERT)**:
    - Kích hoạt khi `use_offsets=False`.
    - Sử dụng `align_spans_to_token_labels_phobert`: Kết hợp tách từ (Word Segmentation) trước khi tokenize.

- **Kết quả (`ds_tok`)**:
  - Dataset đã được mã hóa thành các tensor số học.
  - Chứa đầy đủ các trường: `input_ids`, `attention_mask`, `labels` (nhãn cứng) và `soft_labels` (phân phối xác suất).

In [27]:
MODEL_XLMR = "xlm-roberta-base"
MODEL_PHOBERT = "vinai/phobert-base"
MODEL_PHOBERT_LARGE = "vinai/phobert-large"
MODEL_MICROSOFT_NER = "microsoft/deberta-v3-large"

# model_name = MODEL_PHOBERT
# model_name = MODEL_XLMR
# model_name = MODEL_PHOBERT_LARGE
# Cấu hình tên mô hình sẽ sử dụng để huấn luyện.
model_name = MODEL_MICROSOFT_NER

# tokenizer dùng để TRAIN + pad
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
use_offsets = has_offset_mapping(tokenizer)
print("TRAIN MODEL:", model_name)
print("Tokenizer class:", type(tokenizer))
print("Has offset_mapping:", use_offsets)

if use_offsets:
    # XLM-R / GGPT-NER
    ds_tok = ds.map(
        lambda ex: align_spans_to_token_soft_labels(tokenizer, label2id, ex),
        batched=True,
        desc=f"Tokenize + Align spans→token labels (offsets) [{model_name}]",
    )
else:
    # PhoBERT
    ds_tok = ds.map(
        lambda ex: align_spans_to_token_labels_phobert(tokenizer, label2id, ex, max_length=256),
        batched=True,
        desc=f"Tokenize + Align spans→token labels (wordseg) [{model_name}]",
    )

print(ds_tok)
print(ds_tok["train"][0].keys())

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


TRAIN MODEL: microsoft/deberta-v3-large
Tokenizer class: <class 'transformers.models.deberta_v2.tokenization_deberta_v2_fast.DebertaV2TokenizerFast'>
Has offset_mapping: True


Tokenize + Align spans→token labels (offsets) [microsoft/deberta-v3-large]:   0%|          | 0/2411 [00:00<?, …

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokenize + Align spans→token labels (offsets) [microsoft/deberta-v3-large]:   0%|          | 0/301 [00:00<?, ?…

Tokenize + Align spans→token labels (offsets) [microsoft/deberta-v3-large]:   0%|          | 0/301 [00:00<?, ?…

DatasetDict({
    train: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'soft_labels'],
        num_rows: 2411
    })
    validation: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'soft_labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'soft_labels'],
        num_rows: 301
    })
})
dict_keys(['text', 'spans_start', 'spans_end', 'spans_label', 'spans_prob', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'soft_labels'])


### Sanity check độ dài input & nhãn

- Lấy một mẫu từ tập **train** đã tokenize và căn nhãn.
- In ra độ dài:
  - `input_ids`
  - `labels`
- **Assert** đảm bảo:
  ```text
  len(input_ids) == len(labels)
  ```
→ mỗi token đều có nhãn tương ứng (hoặc -100 cho token bỏ qua).

In [28]:
ex = ds_tok["train"][0]
print(len(ex["input_ids"]), len(ex["labels"]))
assert len(ex["input_ids"]) == len(ex["labels"])

72 72


### Kiểm tra & Minh họa Căn chỉnh Nhãn (Debug Alignment)

- `_topk_prob(...)`: Hàm tiện ích trích xuất $k$ nhãn có xác suất cao nhất từ vector phân phối, giúp giải mã **Soft Labels **thành dạng dễ đọc cho con người.

- **Quy trình Kiểm tra (Debug Flow)**:
  - Lấy 1 mẫu thực tế từ tập huấn luyện (`train[0]`).
  - Thực hiện **Tokenize & Alignment** (tương tự quy trình huấn luyện).
  - In chi tiết từng token kèm các thông số:
    - **Token & Offset**: Từ đã tách và vị trí trong văn bản gốc.
    - **Hard Label**: Nhãn BIO cứng (dùng để so sánh).
    - **Soft Top-2**: Hai nhãn có xác suất cao nhất (để kiểm chứng xem mô hình có nhận được thông tin xác suất từ dữ liệu đầu vào hay không).

- **Mục đích**: Xác minh trực quan ("Sanity Check") độ chính xác của thuật toán căn chỉnh, đặc biệt tại các vị trí ranh giới thực thể hoặc nơi có sự nhập nhằng.

In [ ]:


def _as_prob_dict(x):
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            return json.loads(x)
        except Exception:
            return {}
    return {}

def _topk_prob(vec, id2label, k=2):
    """vec: list[float] length C; returns list[(label,prob)] top-k."""
    arr = np.array(vec, dtype=float)
    if arr.size == 0:
        return []
    idx = arr.argsort()[-k:][::-1]
    return [(id2label[int(i)], float(arr[int(i)])) for i in idx]

# Debug 1 sample
sample = ds["train"][0]
sample_text = sample["text"]
starts = sample["spans_start"]
ends   = sample["spans_end"]
labs   = sample["spans_label"]
probs  = sample.get("spans_prob", [])  # JSON strings

print("TEXT:", sample_text)
print("SPANS:", list(zip(starts, ends, labs)))
print("HAS spans_prob:", len(probs) == len(labs))
print("-" * 80)

if has_offset_mapping(tokenizer):
    # XLM-R / fast tokenizer
    enc_dbg = tokenizer(sample_text, return_offsets_mapping=True, truncation=True, max_length=512)
    offsets = enc_dbg["offset_mapping"]
    tokens  = tokenizer.convert_ids_to_tokens(enc_dbg["input_ids"])

    # use SOFT align (new)
    one = align_spans_to_token_soft_labels(
        tokenizer, label2id,
        {
            "text": [sample_text],
            "spans_start": [starts],
            "spans_end": [ends],
            "spans_prob": [probs],   # JSON strings OK (align has parse)
        }
    )

    label_ids = one["labels"][0]
    soft_vecs = one["soft_labels"][0]

    for tok, (s, e), lid, svec in zip(tokens, offsets, label_ids, soft_vecs):
        hard = "IGN" if lid == -100 else id2label[int(lid)]
        piece = "" if (s == e) else sample_text[s:e]

        if lid == -100:
            top2 = []
        else:
            top2 = _topk_prob(svec, id2label, k=2)

        print(f"{tok:18s}  offset=({s:3d},{e:3d})  text='{piece}'  hard={hard}  soft_top2={top2}")

else:
    one = align_spans_to_token_labels_phobert(
        tokenizer, label2id,
        {
            "text": [sample_text],
            "spans_start": [starts],
            "spans_end": [ends],
            "spans_label": [labs],   # PhoBERT hiện chỉ hard
        },
        max_length=256
    )

    tokens = tokenizer.convert_ids_to_tokens(one["input_ids"][0])
    label_ids = one["labels"][0]

    for tok, lid in zip(tokens, label_ids):
        hard = "IGN" if lid == -100 else id2label[int(lid)]
        print(f"{tok:18s}  hard={hard}")



TEXT: Về cơ quan sinh sản Căn cứ vào 2 dạng chính là bào tử và hạt để chia thành 2 nhóm là thực vật sinh sản bằng bào tử hay bằng hạt.
SPANS: [(3, 19, 'PLANT_PART'), (47, 53, 'PLANT_PART'), (57, 60, 'PLANT_PART'), (108, 114, 'PLANT_PART'), (124, 127, 'PLANT_PART')]
HAS spans_prob: True
--------------------------------------------------------------------------------
[CLS]               offset=(  0,  0)  text=''  hard=IGN  soft_top2=[]
▁V                  offset=(  0,  1)  text='V'  hard=O  soft_top2=[('O', 1.0), ('B-SYMPTOM', 0.0)]
ề                   offset=(  1,  2)  text='ề'  hard=O  soft_top2=[('O', 1.0), ('B-SYMPTOM', 0.0)]
▁c                  offset=(  2,  4)  text=' c'  hard=B-PLANT_PART  soft_top2=[('B-PLANT_PART', 0.89399), ('B-HERB', 0.10601)]
ơ                   offset=(  4,  5)  text='ơ'  hard=I-PLANT_PART  soft_top2=[('I-PLANT_PART', 0.89399), ('I-HERB', 0.10601)]
▁quan               offset=(  5, 10)  text=' quan'  hard=I-PLANT_PART  soft_top2=[('I-PLANT_PART', 0.89399), ('

## Khởi tạo mô hình Token Classification cho NER

- `AutoModelForTokenClassification.from_pretrained(...)`:
  - Tải trọng số (weights) của backbone đã huấn luyện trước (`model_name`).
  - Tự động khởi tạo và gắn thêm lớp **Classifier Head** (lớp tuyến tính) ở tầng cuối cùng để phục vụ việc phân loại từng token.
  
- **Tham số cấu hình**:
  - `num_labels`: Định nghĩa kích thước đầu ra của lớp Classifier, bằng tổng số nhãn trong hệ BIO (`len(LABELS)`).
  - `id2label` / `label2id`: Tích hợp trực tiếp bảng ánh xạ vào cấu hình mô hình (`config`), giúp thuận tiện cho việc suy luận (inference) và giải mã nhãn sau này mà không cần file map rời.

- **Mục đích**:
  - Chuẩn bị kiến trúc mô hình sẵn sàng cho bài toán **Sequence Labeling** (Gán nhãn chuỗi).
  - Đảm bảo không gian vector đầu ra (logits) khớp chính xác với hệ nhãn dữ liệu đã xử lý.

In [30]:
model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

Some weights of DebertaV2ForTokenClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## TrainingArguments + Trainer

### Data Collator tùy chỉnh cho Soft Labels

- `SoftLabelDataCollator`:
  - Lớp Collator tùy biến kế thừa từ `DataCollatorForTokenClassification`.
  - Giải quyết vấn đề **padding đồng bộ** cho cả nhãn cứng (integer) và nhãn mềm (probability matrix) trong cùng một batch.

- **Cơ chế hoạt động (`__call__`)**:
  - **Tách dữ liệu**: Tạm thời rút `soft_labels` ra khỏi input để tránh lỗi khi gọi collator mặc định của HuggingFace.
  - **Padding chuẩn**: Gọi `super().__call__` để pad `input_ids`, `labels` (nhãn cứng) theo độ dài chuỗi lớn nhất trong batch (`seq_len`).
  - **Padding Soft Labels**: Thủ công thêm các vector `[0.0, ...]` vào `soft_labels` để khớp chiều dài với `seq_len` của batch vừa tạo.
  - **Tensor hóa**: Chuyển đổi danh sách kết quả về `torch.float32` tensor và đóng gói lại vào batch.

- **Mục đích**:
  - Đảm bảo ma trận `soft_labels` có kích thước chuẩn (`batch_size`, `seq_len`, `num_labels`), khớp tuyệt đối với đầu ra của mô hình để tính hàm mất mát phân phối (KL Divergence).

In [31]:

class SoftLabelDataCollator(DataCollatorForTokenClassification):
    def __call__(self, features):
        # tách soft_labels ra trước để HF pad input_ids/labels như bình thường
        soft = [f.pop("soft_labels", None) for f in features]

        batch = super().__call__(features)  # pad input_ids, attention_mask, labels

        # nếu không có soft_labels => return như cũ
        if all(s is None for s in soft):
            return batch

        # pad soft_labels theo seq_len batch
        seq_len = batch["input_ids"].shape[1]
        num_labels = None

        soft_out = []
        labels = batch.get("labels", None)

        for i, s in enumerate(soft):
            if s is None:
                # không có => fill 0
                if num_labels is None:
                    # đoán từ model sau cũng được, nhưng ở đây lấy từ labels max không chuẩn
                    # => ta suy từ sample khác có soft_labels
                    pass
                soft_out.append(None)
                continue

            # s: list[list[float]] length = unpadded seq
            if num_labels is None and len(s) > 0:
                num_labels = len(s[0])

            # pad/trunc
            s = s[:seq_len]
            if len(s) < seq_len:
                pad_vec = [0.0] * (num_labels if num_labels is not None else 1)
                s = s + [pad_vec] * (seq_len - len(s))

            soft_out.append(s)

        # nếu num_labels vẫn None (vd batch toàn None/empty) => bỏ qua
        if num_labels is None:
            return batch

        # fill những sample soft_out=None => zeros
        for i in range(len(soft_out)):
            if soft_out[i] is None:
                soft_out[i] = [[0.0]*num_labels for _ in range(seq_len)]

        batch["soft_labels"] = torch.tensor(soft_out, dtype=torch.float32)
        return batch

### Trainer tùy chỉnh cho Soft Labels (KL-Divergence Loss)

- `SoftLabelTrainer(Trainer)`:
  - Kế thừa lớp `Trainer` chuẩn của HuggingFace để can thiệp trực tiếp vào bước tính toán hàm mất mát (`compute_loss`).

- **Cơ chế hoạt động**:
  - **Fallback thông minh**:
    - Nếu batch không có `soft_labels` $\to$ Sử dụng hàm loss mặc định của mô hình (CrossEntropy trên nhãn cứng).
    - Nếu có `soft_labels` $\to$ Chuyển sang tính **KL Divergence**.
  - **Masking (-100)**: Sử dụng `inputs["labels"]` để tạo mask, loại bỏ các token đặc biệt (`[CLS]`, `[SEP]`, padding) khỏi quá trình tính toán loss, đảm bảo mô hình chỉ học trên các token có nghĩa.
  - **Công thức Loss**: $$D_{KL}(P || Q) = \sum P(x) \log \left( \frac{P(x)}{Q(x)} \right)$$
    - **Input**: `log_softmax(logits)` (Phân phối dự đoán).
    - **Target**: `soft_labels` (Phân phối xác suất thực tế từ dữ liệu).

- **Mục đích**:
  - Cho phép mô hình học được độ mờ và sự lưỡng lự của dữ liệu (ví dụ: 80% là Bệnh, 20% là Triệu chứng) thay vì ép buộc về một nhãn cứng duy nhất, giúp tăng độ bền vững khi gặp các thực thể nhập nhằng.

In [32]:
class SoftLabelTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        soft = inputs.pop("soft_labels", None)

        outputs = model(**inputs)
        logits = outputs.logits  # (B, L, C)

        # nếu không có soft_labels => dùng loss gốc của model (CE với inputs["labels"])
        if soft is None:
            loss = outputs.loss
            return (loss, outputs) if return_outputs else loss

        # mask theo labels!=-100 để bỏ special/pad
        labels = inputs.get("labels", None)
        if labels is None:
            # không có labels thì coi như tất cả token đều valid
            mask = torch.ones(logits.shape[:2], dtype=torch.bool, device=logits.device)
        else:
            mask = (labels != -100)

        # KLDiv: target là phân phối (soft), input là log-prob
        log_probs = F.log_softmax(logits, dim=-1)          # (B,L,C)
        target = soft.to(logits.device)                    # (B,L,C)

        # chỉ lấy token valid
        log_probs = log_probs[mask]
        target = target[mask]

        # nếu batch toàn token bị mask => loss=0 (tránh NaN)
        if log_probs.numel() == 0:
            loss = logits.sum() * 0.0
            return (loss, outputs) if return_outputs else loss

        loss = F.kl_div(log_probs, target, reduction="batchmean")
        return (loss, outputs) if return_outputs else loss


### Cấu hình tham số & Khởi tạo Trainer

- `TrainingArguments`: Thiết lập các siêu tham số quan trọng cho quá trình Fine-tuning:
  - **Tối ưu tài nguyên**: `fp16=True` (Mixed Precision) giúp giảm bộ nhớ VRAM và tăng tốc độ huấn luyện.
  - **Chiến lược lưu & đánh giá**: Thực hiện theo từng epoch; tự động tải lại mô hình tốt nhất (`load_best_model_at_end=True`) dựa trên chỉ số F1 cao nhất.
  - **Hyperparameters**: `learning_rate=2e-5`, `batch_size=4`, `num_train_epochs=5`.

- `SoftLabelDataCollator`:
  - Khởi tạo bộ gom dữ liệu tùy chỉnh để xử lý batch.
  - Đảm bảo việc padding đồng bộ cho cả `input_ids` và ma trận `soft_labels`.

- `SoftLabelTrainer`:
  - Tổng hợp các thành phần: Mô hình, Dữ liệu (Train/Valid), Collator và Hàm đánh giá (`compute_metrics`).
  - Đóng vai trò trình điều khiển chính, sử dụng hàm Loss KL-Divergence đã tùy biến để huấn luyện mô hình.

In [ ]:
from transformers import DataCollatorForTokenClassification

OUTPUT_DIR_MAP = {
    MODEL_XLMR: "ner_xlmr_soft",
    MODEL_PHOBERT: "ner_phobert_soft",
    MODEL_PHOBERT_LARGE: "ner_phobert_large_soft",
    MODEL_MICROSOFT_NER: "ner_deberta_v3_large_soft", 
}

current_output_dir = OUTPUT_DIR_MAP.get(model_name, "ner_custom_soft")

args = TrainingArguments(
    output_dir=current_output_dir,
    eval_strategy="epoch",
    logging_strategy="steps",
    save_strategy="epoch",
    logging_steps=200,
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=True,
)

data_collator = SoftLabelDataCollator(
    tokenizer=tokenizer,
    padding=True,
    label_pad_token_id=-100
)

trainer = SoftLabelTrainer(
    model=model,
    args=args,
    train_dataset=ds_tok["train"],
    eval_dataset=ds_tok["validation"],
    data_collator=data_collator,
    compute_metrics=build_metrics_fn(id2label),
)

### Sanity Check dữ liệu trước khi huấn luyện

- **Kiểm tra đồng bộ độ dài (Token-Label Alignment)**:
  - Duyệt qua `n` mẫu đầu tiên trong tập `train` và `validation`.
  - Xác nhận điều kiện `len(input_ids) == len(labels)`. → Đảm bảo mỗi token đầu vào đều được gán đúng một nhãn tương ứng, không bị lệch sau bước tokenize.

- **Kiểm tra miền giá trị nhãn (Label Range)**:
  - Bỏ qua giá trị `-100` (dùng cho padding hoặc special tokens).
  - Các nhãn hợp lệ phải nằm trong khoảng `[0, NUM_LABELS)`. → Đảm bảo dữ liệu không chứa nhãn lạ (out-of-bound) so với hệ nhãn BIO đã định nghĩa.

- **Kiểm tra tương thích Vocab (Tokenizer vs Model)**:
  - Tìm `input_id` lớn nhất trong toàn bộ tập huấn luyện.
  - Đảm bảo giá trị này nhỏ hơn `model.config.vocab_size`. → Ngăn chặn lỗi **IndexError** tại lớp Embedding do sử dụng nhầm Tokenizer không khớp với cấu hình Backbone mô hình.

Các bước này giúp phát hiện sớm các lỗi kỹ thuật nghiêm trọng (lệch nhãn, sai config) trước khi bắt đầu quá trình huấn luyện tốn kém.

In [34]:
NUM_LABELS = len(LABELS)
def sanity_check(split, name, n=50):
    for i in range(min(n, len(split))):
        x = split[i]["input_ids"]
        y = split[i]["labels"]
        assert len(x) == len(y), (name, i, len(x), len(y))
        for v in y:
            if v == -100:
                continue
            assert 0 <= int(v) < NUM_LABELS, (name, i, v)
    print("OK:", name)

sanity_check(ds_tok["train"], "train")
sanity_check(ds_tok["validation"], "validation")

# vocab check
mx = max(max(row["input_ids"]) for row in ds_tok["train"])
print("max input_id:", mx, "| vocab_size:", model.config.vocab_size)
assert mx < model.config.vocab_size, "input_ids vượt vocab => tokenizer/model mismatch"

OK: train
OK: validation
max input_id: 127982 | vocab_size: 128100


## Training

In [35]:
trains_stats = trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.416800,0.355750,0.511283,0.616321,0.558909
2,0.313700,0.289802,0.647574,0.687903,0.667129
3,0.235500,0.289438,0.645431,0.717967,0.679770
4,0.162300,0.287211,0.690318,0.745168,0.716695
5,0.126600,0.304605,0.690993,0.752326,0.720356


## Lưu Best Checkpoint (Model + Tokenizer + Label Maps)

- **Thiết lập thư mục lưu trữ**:
  - Định nghĩa đường dẫn gốc `BASE_CKPT_DIR` trên Google Drive (`.../checkpointNewMethod`) để lưu trữ kết quả bền vững.

- **Quản lý phiên bản theo Backbone (`MODEL2RUN`)**:
  - Mỗi kiến trúc mô hình (XLM-R / PhoBERT / DeBERTa) được gán một tên định danh riêng (subfolder).
  - Mục đích: Tránh ghi đè checkpoint giữa các lần thử nghiệm khác nhau và dễ dàng so sánh hiệu năng.

- **safe_slug(model_name)**:
  - Hàm tiện ích giúp làm sạch tên mô hình (thay thế `/` thành `_`).
  - Đảm bảo tạo được tên thư mục hợp lệ ngay cả khi sử dụng model mới chưa có trong từ điển cấu hình.

- **Lưu trữ trọn bộ Artefacts**:
  - `trainer.save_model(...)`: Lưu trọng số (weights) và cấu hình (config) của mô hình tốt nhất.
  - `tokenizer.save_pretrained(...)`: Lưu các tệp tin của Tokenizer (vocab, merges) để đảm bảo đồng bộ khi tách từ.
  - `label_maps.json`: Lưu cấu hình ánh xạ nhãn (`LABELS`, `label2id`, `id2label`). Đây là file bắt buộc để giải mã output (ID $\to$ BIO tag) trong quá trình Inference sau này.

- **Kết quả**: In ra đường dẫn tuyệt đối đến thư mục checkpoint đã lưu thành công.

In [ ]:
import os, json

# Base checkpoint directory (Google Drive)
BASE_CKPT_DIR = "/content/drive/MyDrive/medical_ner/checkpoint"

# Map model → subdir
MODEL2RUN = {
    MODEL_XLMR: "ner_xlmr_soft",
    MODEL_PHOBERT: "ner_phobert_soft",
    MODEL_PHOBERT_LARGE: "ner_phobert_large_soft",
    MODEL_MICROSOFT_NER: "ner_deberta_v3_large_soft",
}

def safe_slug(s: str) -> str:
    # fallback: biến "microsoft/deberta-v3-large" -> "microsoft_deberta-v3-large"
    return (s or "unknown").replace("/", "_").replace(" ", "_")

run_name = MODEL2RUN.get(model_name, f"ner_{safe_slug(model_name)}")

# Final best checkpoint dir
best_dir = os.path.join(BASE_CKPT_DIR, run_name)
os.makedirs(best_dir, exist_ok=True)

trainer.save_model(best_dir)
tokenizer.save_pretrained(best_dir)

with open(os.path.join(best_dir, "label_maps.json"), "w", encoding="utf-8") as f:
    json.dump({"labels": LABELS, "label2id": label2id, "id2label": id2label},
              f, ensure_ascii=False, indent=2)

print("Saved to:", best_dir)

Saved to: /content/drive/MyDrive/checkpointNewMethod/ner_deberta_v3_large


# **Evaluate**

## Đánh giá & Xuất kết quả từ Checkpoint

- **Tính toán Metrics (`make_compute_metrics`, `make_entity_df`)**:
  - Tính toán các chỉ số **Precision, Recall, F1-score** ở mức thực thể (Entity-level) thay vì token-level.
  - Tạo bảng báo cáo chi tiết cho từng loại nhãn (VD: `HERB`, `DISEASE`) để dễ dàng phân tích điểm mạnh/yếu của mô hình.

- **Tái lập thực thể dự đoán (Span Reconstruction)**:
  - Chuyển đổi chuỗi nhãn dự đoán (Tokens) trở lại dạng span ký tự (`start`, `end`) trên văn bản gốc.
  - **Nhánh Fast Tokenizer**: Dùng `offset_mapping` để ánh xạ trực tiếp (chính xác cao).
  - **Nhánh PhoBERT**: Dùng cơ chế ánh xạ qua `word_ids` (Word Segmentation).

- **Xuất kết quả (`export_predictions_to_jsonl_compact_unified`)**:
  - Tổng hợp thông tin: Văn bản gốc, Nhãn thực tế (Ground Truth) và Nhãn dự đoán kèm độ tin cậy (Confidence Score).
  - Lưu trữ dưới dạng JSONL giúp dễ dàng truy xuất, kiểm tra sai sót (Error Analysis) hoặc tích hợp vào các công cụ Visualization.

- **Pipeline đánh giá tổng thể (evaluate_from_checkpoint)**:
  - Hàm điều khiển trung tâm: Tự động tải Model & Tokenizer từ đường dẫn checkpoint.
  - Thực thi quy trình khép kín:  `Tokenize Test Set` $\to$ `Predict` $\to$ `Calculate Metrics` $\to$ `Export Report`.

In [ ]:
def softmax_np(x, axis=-1):
    """Tính toán hàm Softmax bằng numpy để chuyển đổi Logits thành xác suất [0, 1]."""
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def nfc(text: str) -> str:
    import unicodedata
    return unicodedata.normalize("NFC", text or "")

def has_offset_mapping(tokenizer) -> bool:
    return bool(getattr(tokenizer, "is_fast", False))

def _as_prob_dict(x):
    """
    spans_prob item can be dict or JSON string. Return dict.
    """
    if x is None:
        return {}
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return {}
        try:
            return json.loads(x)
        except Exception:
            return {}
    return {}

def make_compute_metrics(id2label):
    """
    Khởi tạo hàm compute_metrics cho Trainer để tính Precision, Recall, F1.
    Hàm này tự động loại bỏ nhãn -100 và chuyển ID thành tên nhãn thực thể.
    """
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        true_labels, true_preds = [], []
        for p_row, l_row in zip(preds, labels):
            sent_true, sent_pred = [], []
            for p, l in zip(p_row, l_row):
                if l == -100:
                    continue
                sent_true.append(id2label[int(l)])
                sent_pred.append(id2label[int(p)])
            true_labels.append(sent_true)
            true_preds.append(sent_pred)

        return {
            "precision": precision_score(true_labels, true_preds),
            "recall": recall_score(true_labels, true_preds),
            "f1": f1_score(true_labels, true_preds),
        }
    return compute_metrics

def show_overall_metrics(metrics: dict):
    keys = ["eval_precision", "eval_recall", "eval_f1", "eval_loss"]
    print("\nOVERALL TEST METRICS")
    for k in keys:
        if k in metrics:
            print(f"{k.replace('eval_', '').upper():10s}: {metrics[k]:.4f}")

def make_entity_df(true_labels, true_preds):
    """
    Tạo bảng báo cáo (DataFrame) chi tiết hiệu năng cho từng loại thực thể riêng biệt.
    Giúp phân tích xem mô hình đang yếu hay mạnh ở nhãn nào (DISEASE, SYMPTOM,...).
    """
    rep = classification_report(
        true_labels,
        true_preds,
        output_dict=True,
        zero_division=0,
    )

    rows = []
    for label, scores in rep.items():
        if label in ("micro avg", "macro avg", "weighted avg"):
            continue
        rows.append({
            "Entity": label,
            "Precision": float(scores["precision"]),
            "Recall": float(scores["recall"]),
            "F1": float(scores["f1-score"]),
            "Support": int(scores["support"]),
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df
    return df.sort_values(["Support", "F1"], ascending=[False, False]).reset_index(drop=True)

def align_spans_to_token_soft_labels(tokenizer, label2id, examples):
    texts = examples["text"]
    starts_batch = examples["spans_start"]
    ends_batch   = examples["spans_end"]
    probs_batch  = examples.get("spans_prob", None)

    enc = tokenizer(
        texts,
        truncation=True,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )

    num_labels = len(label2id)
    o_id = label2id["O"]

    all_hard = []
    all_soft = []

    # if spans_prob missing, create empty lists
    if probs_batch is None:
        probs_batch = [[] for _ in range(len(texts))]

    for text, starts, ends, offsets, span_probs in zip(
        texts, starts_batch, ends_batch, enc["offset_mapping"], probs_batch
    ):
        text = text or ""
        span_probs = span_probs or []

        # init per token: O=1
        token_hard = []
        token_soft = []
        for (s, e) in offsets:
            if s == e:
                token_hard.append(-100)
                token_soft.append([0.0] * num_labels)
            else:
                token_hard.append(o_id)
                v = [0.0] * num_labels
                v[o_id] = 1.0
                token_soft.append(v)

        # apply spans soft
        for ss, ee, prob_x in zip(starts, ends, span_probs):
            ss = int(ss); ee = int(ee)
            prob_dict = _as_prob_dict(prob_x)

            first = True
            for i, (os, oe) in enumerate(offsets):
                if os == oe:
                    continue
                if oe <= ss or os >= ee:
                    continue

                vec = [0.0] * num_labels
                for typ, p in prob_dict.items():
                    if p is None:
                        continue
                    tag = ("B-" if first else "I-") + str(typ)
                    if tag in label2id:
                        vec[label2id[tag]] += float(p)

                if sum(vec) <= 0:
                    vec[o_id] = 1.0
                else:
                    z = sum(vec)
                    vec = [x / z for x in vec]

                token_soft[i] = vec
                token_hard[i] = int(np.argmax(vec))
                first = False

        all_hard.append(token_hard)
        all_soft.append(token_soft)

    enc.pop("offset_mapping")
    enc["labels"] = all_hard
    enc["soft_labels"] = all_soft
    return enc

def viet_words_with_offsets(text: str):
    text = nfc(text)
    words = text.split()
    out = []
    cur = 0
    for w in words:
        pos = text.find(w, cur)
        if pos == -1:
            continue
        s = pos
        e = pos + len(w)
        out.append((w, s, e))
        cur = e
    return out

def build_char_mask(text: str, spans):
    mask = [None] * len(text)
    for s, e, lab in spans:
        s = max(0, min(int(s), len(text)))
        e = max(0, min(int(e), len(text)))
        for i in range(s, e):
            mask[i] = lab
    return mask

def word_bio_tags(text: str, spans):
    text = nfc(text)
    words_info = viet_words_with_offsets(text)
    mask = build_char_mask(text, spans)

    tags = []
    for (w, ws, we) in words_info:
        lab = None
        for i in range(ws, min(we, len(mask))):
            if mask[i] is not None:
                lab = mask[i]
                break

        if lab is None:
            tags.append("O")
        else:
            is_begin = (ws < len(mask) and mask[ws] == lab and (ws == 0 or mask[ws-1] != lab))
            tags.append(f"B-{lab}" if is_begin else f"I-{lab}")
    return words_info, tags

def encode_words_and_labels(tok, words, word_tags, label2id, max_length=256):
    input_ids = []
    labels = []

    for w, tag in zip(words, word_tags):
        ids = tok.encode(w, add_special_tokens=False)
        if len(ids) == 0:
            continue

        if tag == "O":
            input_ids.extend(ids)
            labels.extend([label2id["O"]] * len(ids))
        else:
            input_ids.append(ids[0])
            labels.append(label2id[tag])

            itag = ("I-" + tag[2:]) if tag.startswith("B-") else tag
            input_ids.extend(ids[1:])
            labels.extend([label2id[itag]] * (len(ids) - 1))

        if len(input_ids) >= (max_length - 2):
            input_ids = input_ids[: (max_length - 2)]
            labels = labels[: (max_length - 2)]
            break

    input_ids = tok.build_inputs_with_special_tokens(input_ids)
    labels = [-100] + labels + [-100]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

def align_spans_to_token_labels_phobert(tokenizer, label2id, examples, max_length=256):
    out = {"input_ids": [], "attention_mask": [], "labels": []}

    for text, starts, ends, labs in zip(
        examples["text"], examples["spans_start"], examples["spans_end"], examples["spans_label"]
    ):
        text = nfc(text)
        spans = [(s, e, l) for s, e, l in zip(starts, ends, labs)]

        words_info, word_tags = word_bio_tags(text, spans)
        words = [w for (w, _, _) in words_info]

        enc = encode_words_and_labels(tokenizer, words, word_tags, label2id, max_length=max_length)
        out["input_ids"].append(enc["input_ids"])
        out["attention_mask"].append(enc["attention_mask"])
        out["labels"].append(enc["labels"])

    return out

def build_ds_tok_for_model(ds, tokenizer, label2id, model_name, max_length_phobert=256):
    use_offsets = has_offset_mapping(tokenizer)
    print("EVAL MODEL:", model_name)
    print("Tokenizer class:", type(tokenizer))
    print("Has offset_mapping:", use_offsets)

    if use_offsets:
        ds_tok = ds.map(
            lambda ex: align_spans_to_token_soft_labels(tokenizer, label2id, ex),
            batched=True,
            desc=f"Tokenize + Align spans→token SOFT labels (offsets) [{model_name}]",
        )
    else:
        ds_tok = ds.map(
            lambda ex: align_spans_to_token_labels_phobert(
                tokenizer, label2id, ex, max_length=max_length_phobert
            ),
            batched=True,
            desc=f"Tokenize + Align spans→token labels (wordseg) [{model_name}]",
        )
    return ds_tok

def tags_to_spans_with_score(offsets, tags, confs):
    spans = []
    cur_type, cur_s, cur_e = None, None, None
    cur_scores = []

    def flush():
        nonlocal cur_type, cur_s, cur_e, cur_scores
        if cur_type is not None and cur_s is not None and cur_e is not None and cur_e > cur_s:
            spans.append([int(cur_s), int(cur_e), cur_type, float(np.mean(cur_scores)) if cur_scores else 0.0])
        cur_type, cur_s, cur_e, cur_scores = None, None, None, []

    for (s, e), tag, sc in zip(offsets, tags, confs):
        if s is None or e is None or s == e:
            continue

        if tag is None or tag == "O":
            flush()
            continue

        if tag.startswith("B-"):
            flush()
            cur_type = tag[2:]
            cur_s, cur_e = s, e
            cur_scores = [float(sc)]
        elif tag.startswith("I-"):
            typ = tag[2:]
            if cur_type != typ or cur_type is None:
                flush()
                cur_type = typ
                cur_s, cur_e = s, e
                cur_scores = [float(sc)]
            else:
                cur_e = max(cur_e, e)
                cur_scores.append(float(sc))
        else:
            flush()

    flush()
    return spans

def build_pred_spans_fast(text, pred_ids, pred_conf, tok, id2label, max_len_tokens=None):
    enc = tok(
        text,
        truncation=True,
        max_length=max_len_tokens,
        return_offsets_mapping=True,
        add_special_tokens=True,
    )
    offsets = enc["offset_mapping"]
    tags = [id2label[int(x)] for x in pred_ids]
    confs = [float(x) for x in pred_conf]
    L = min(len(offsets), len(tags), len(confs))
    return tags_to_spans_with_score(offsets[:L], tags[:L], confs[:L])

def encode_words_and_labels_with_map(tok, words, max_length=256):
    input_ids = []
    word_ids = []

    for wi, w in enumerate(words):
        ids = tok.encode(w, add_special_tokens=False)
        if len(ids) == 0:
            continue
        for tid in ids:
            input_ids.append(tid)
            word_ids.append(wi)
        if len(input_ids) >= (max_length - 2):
            input_ids = input_ids[: (max_length - 2)]
            word_ids = word_ids[: (max_length - 2)]
            break

    input_ids = tok.build_inputs_with_special_tokens(input_ids)
    word_ids = [None] + word_ids + [None]
    attention_mask = [1] * len(input_ids)
    return {"input_ids": input_ids, "attention_mask": attention_mask, "word_ids": word_ids}

def word_tags_from_token_preds(word_ids, pred_ids, pred_conf, id2label):
    tmp = {}
    for wi, pid, cf in zip(word_ids, pred_ids, pred_conf):
        if wi is None:
            continue
        tmp.setdefault(wi, []).append((id2label[int(pid)], float(cf)))

    max_wi = max(tmp.keys()) if tmp else -1
    word_tags, word_confs = [], []
    for wi in range(max_wi + 1):
        arr = tmp.get(wi, [])
        if not arr:
            word_tags.append("O")
            word_confs.append(0.0)
            continue
        word_tags.append(arr[0][0])
        word_confs.append(float(np.mean([x[1] for x in arr])))
    return word_tags, word_confs

def build_pred_spans_phobert(text, pred_ids, pred_conf, tok, id2label, max_len_tokens=256):
    text = nfc(text)
    words_info = viet_words_with_offsets(text)
    words = [w for (w, _, _) in words_info]

    enc = encode_words_and_labels_with_map(tok, words, max_length=max_len_tokens)
    word_ids = enc["word_ids"]

    L = min(len(word_ids), len(pred_ids), len(pred_conf))
    word_ids  = word_ids[:L]
    pred_ids  = pred_ids[:L]
    pred_conf = pred_conf[:L]

    word_tags, word_confs = word_tags_from_token_preds(word_ids, pred_ids, pred_conf, id2label)

    spans = []
    cur_type, cur_s, cur_e = None, None, None
    cur_scores = []

    def flush():
        nonlocal cur_type, cur_s, cur_e, cur_scores
        if cur_type is not None and cur_s is not None and cur_e is not None and cur_e > cur_s:
            spans.append([int(cur_s), int(cur_e), cur_type, float(np.mean(cur_scores)) if cur_scores else 0.0])
        cur_type, cur_s, cur_e, cur_scores = None, None, None, []

    for (w, ws, we), tag, cf in zip(words_info, word_tags, word_confs):
        if tag is None or tag == "O":
            flush()
            continue
        if tag.startswith("B-"):
            flush()
            cur_type = tag[2:]
            cur_s, cur_e = ws, we
            cur_scores = [float(cf)]
        elif tag.startswith("I-"):
            typ = tag[2:]
            if cur_type != typ or cur_type is None:
                flush()
                cur_type = typ
                cur_s, cur_e = ws, we
                cur_scores = [float(cf)]
            else:
                cur_e = max(cur_e, we)
                cur_scores.append(float(cf))
        else:
            flush()

    flush()
    return spans

def export_predictions_to_jsonl_compact_unified(
    raw_test,
    ds_tok_test,
    preds,
    pred_prob,
    tok,
    id2label,
    out_path="pred_compact.jsonl",
    min_pred=0.0,
):
    fast = has_offset_mapping(tok)

    with open(out_path, "w", encoding="utf-8") as f:
        for i in range(len(raw_test)):
            raw_item = raw_test[i]
            text = raw_item.get("text", "")
            sample_id = raw_item.get("id", i)
            comments = raw_item.get("Comments", [])

            # rebuild GT label from spans_*
            starts = raw_item.get("spans_start", []) or []
            ends   = raw_item.get("spans_end", []) or []
            labs   = raw_item.get("spans_label", []) or []
            probs  = raw_item.get("spans_prob", []) or []

            gt_label = []
            if probs and len(probs) == len(labs):
                for s, e, p in zip(starts, ends, probs):
                    gt_label.append([int(s), int(e), _as_prob_dict(p)])
            else:
                for s, e, lab in zip(starts, ends, labs):
                    gt_label.append([int(s), int(e), {str(lab): 1.0}])

            pred_ids_i = preds[i].tolist()
            conf_i     = pred_prob[i].tolist()

            max_len_tokens = len(ds_tok_test[i]["input_ids"])

            if fast:
                pred_spans = build_pred_spans_fast(
                    text, pred_ids_i, conf_i, tok, id2label, max_len_tokens=max_len_tokens
                )
            else:
                pred_spans = build_pred_spans_phobert(
                    text, pred_ids_i, conf_i, tok, id2label, max_len_tokens=max_len_tokens
                )

            if min_pred is not None and float(min_pred) > 0:
                pred_spans = [sp for sp in pred_spans if float(sp[3]) >= float(min_pred)]

            rec = {
                "id": sample_id,
                "text": text,
                "label": gt_label,
                "Comments": comments,
                "pred": pred_spans,
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

def evaluate_from_checkpoint(
    ds,
    checkpoint_path,
    label2id,
    id2label,
    max_length_phobert=256,
    save_jsonl_path=None,
    min_pred=0.0,
):
    # 1) tokenizer + tokenize dataset
    tok = AutoTokenizer.from_pretrained(checkpoint_path)  # không ép use_fast
    ds_tok_ckpt = build_ds_tok_for_model(
        ds, tok, label2id, model_name=checkpoint_path, max_length_phobert=max_length_phobert
    )

    # 2) model
    model = AutoModelForTokenClassification.from_pretrained(
        checkpoint_path,
        num_labels=len(label2id),
        id2label=id2label,
        label2id=label2id,
    )

    # 3) collator (evaluate chỉ cần pad labels)
    data_collator = DataCollatorForTokenClassification(tokenizer=tok)

    # 4) trainer
    trainer_eval = Trainer(
        model=model,
        tokenizer=tok,  # warning deprecate nhưng vẫn chạy
        data_collator=data_collator,
        compute_metrics=make_compute_metrics(id2label),
    )

    # 5) evaluate overall
    test_metrics = trainer_eval.evaluate(ds_tok_ckpt["test"])
    show_overall_metrics(test_metrics)

    # 6) predict
    preds_output = trainer_eval.predict(ds_tok_ckpt["test"])
    logits = preds_output.predictions          # (N, L, C)
    labels = preds_output.label_ids            # (N, L)
    preds  = np.argmax(logits, axis=-1)        # (N, L)

    p_all = softmax_np(logits, axis=-1)        # (N, L, C)
    pred_prob = np.take_along_axis(p_all, preds[..., None], axis=-1).squeeze(-1)  # (N, L)

    # 7) per-entity report
    true_labels, true_preds = [], []
    for p_row, l_row in zip(preds, labels):
        sent_true, sent_pred = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            sent_true.append(id2label[int(l)])
            sent_pred.append(id2label[int(p)])
        true_labels.append(sent_true)
        true_preds.append(sent_pred)

    df_report = make_entity_df(true_labels, true_preds)
    display(df_report)

    # 8) export JSONL
    if save_jsonl_path is not None:
        export_predictions_to_jsonl_compact_unified(
            raw_test=ds["test"],
            ds_tok_test=ds_tok_ckpt["test"],
            preds=preds,
            pred_prob=pred_prob,
            tok=tok,
            id2label=id2label,
            out_path=save_jsonl_path,
            min_pred=float(min_pred),
        )
        print(f"[Saved] predictions JSONL -> {save_jsonl_path} (min_pred={min_pred})")

    return test_metrics, df_report

## Thực thi Đánh giá & Xuất kết quả dự đoán (Inference)

- **Lựa chọn Checkpoint**:
  - Xác định đường dẫn đến mô hình đã huấn luyện (`checkpoint_path`). Trong trường hợp này là `ner_deberta_v3_large`.

- **Tự động cấu hình tham số**:
  - `is_phobert`: Kiểm tra xem backbone có phải là PhoBERT hay không để thiết lập
  - `max_length` phù hợp (do đặc thù tách từ). Với DeBERTa, tham số này không ảnh hưởng lớn.

- **Quy trình `evaluate_from_checkpoint`**:
  - **Đánh giá tổng quan (Overall Metrics)**: Tính toán Precision, Recall, F1 và Loss trên toàn bộ tập Test.
  - **Báo cáo chi tiết (Per-entity Report):** Tạo DataFrame phân tích hiệu năng trên từng loại thực thể (ví dụ: `HERB` nhận diện tốt hơn `SYMPTOM` hay không?).
  - **Xuất file dự đoán (`pred_compact.jsonl`):** Ghi lại kết quả inference bao gồm tọa độ span, nhãn dự đoán và độ tin cậy trung bình (`avg_conf`) để phục vụ việc phân tích sai số hoặc visualize sau này.

In [ ]:
# --- Pick checkpoint ---
# checkpoint_path = "/content/drive/MyDrive/medical_ner/checkpoint/ner_phobert_soft"
# checkpoint_path = "/content/drive/MyDrive/medical_ner/checkpoint/ner_phobert_large_soft"
# checkpoint_path = "/content/drive/MyDrive/medical_ner/checkpoint/ner_xl
checkpoint_path = "/content/drive/MyDrive/medical_ner/checkpoint/ner_deberta_v3_large_soft"

# --- Auto params (PhoBERT only) ---
is_phobert = ("phobert" in checkpoint_path.lower())
max_length_phobert = 256 if is_phobert else None  # or keep 256, but None makes intent clear

test_metrics, df_report = evaluate_from_checkpoint(
    ds,
    checkpoint_path=checkpoint_path,
    label2id=label2id,
    id2label=id2label,
    max_length_phobert=max_length_phobert if max_length_phobert is not None else 256,
    save_jsonl_path="pred_compact.jsonl",
    min_pred=0,
)

The tokenizer you are loading from '/content/drive/MyDrive/checkpointNewMethod/ner_deberta_v3_large' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


EVAL MODEL: /content/drive/MyDrive/checkpointNewMethod/ner_deberta_v3_large
Tokenizer class: <class 'transformers.models.deberta_v2.tokenization_deberta_v2_fast.DebertaV2TokenizerFast'>
Has offset_mapping: True


Tokenize + Align spans→token SOFT labels (offsets) [/content/drive/MyDrive/checkpointNewMethod/ner_deberta_v3_…

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokenize + Align spans→token SOFT labels (offsets) [/content/drive/MyDrive/checkpointNewMethod/ner_deberta_v3_…

Tokenize + Align spans→token SOFT labels (offsets) [/content/drive/MyDrive/checkpointNewMethod/ner_deberta_v3_…

/tmp/ipython-input-1069922965.py:552: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer_eval = Trainer(



OVERALL TEST METRICS
PRECISION : 0.7023
RECALL    : 0.7414
F1        : 0.7213
LOSS      : 0.3466


,Entity,Precision,Recall,F1,Support
0,PLANT_PART,0.803030,0.830287,0.816431,383
1,HERB,0.784653,0.832021,0.807643,381
2,DISEASE,0.559471,0.582569,0.570787,218
3,SYMPTOM,0.517094,0.573460,0.543820,211
4,HUMAN_PART,0.618182,0.666667,0.641509,153
5,DOSAGE,0.930556,0.917808,0.924138,73


[Saved] predictions JSONL -> pred_compact.jsonl (min_pred=0)


# **Inference**

In [ ]:
def bio_to_spans(text: str, offsets: List[List[int]], pred_ids: List[int], id2label: Dict[int, str]):
    entities = []
    cur = None  # [start, end, TYPE]

    for (s, e), pid in zip(offsets, pred_ids):
        if s == e:
            continue

        tag = id2label[int(pid)]
        if tag == "O":
            if cur:
                entities.append(cur)
                cur = None
            continue

        pref, lab = tag.split("-", 1)

        if pref == "B" or (cur and cur[2] != lab):
            if cur:
                entities.append(cur)
            cur = [s, e, lab]
        else:
            if cur:
                cur[1] = e
            else:
                cur = [s, e, lab]

    if cur:
        entities.append(cur)

    return [{"start": s, "end": e, "label": lab, "text": text[s:e]} for s, e, lab in entities]

## Suy luận NER từ Checkpoint (Inference Pipeline)

- **Khởi tạo & Load Artefacts**:
  - Tải lại `AutoTokenizer` và `AutoModelForTokenClassification` từ thư mục checkpoint (`best_dir`).
  - Đọc file `label_maps.json` để khôi phục bảng ánh xạ nhãn (`id2label`). Chuyển đổi khóa từ chuỗi sang số nguyên (int) để đảm bảo tra cứu chính xác.

- **Cơ chế Tự động phát hiện Tokenizer (`tokenizer_has_offsets`)**:
  - Kiểm tra khả năng hỗ trợ `return_offsets_mapping` của tokenizer.
  - **Nếu có Offsets (Fast Tokenizer)**: Kích hoạt nhánh xử lý cho DeBERTa/XLM-R.
  - **Nếu không (Slow Tokenizer)**: Kích hoạt nhánh xử lý thủ công cho PhoBERT.

- **Nhánh 1: Fast Tokenizer (DeBERTa/XLM-R)**:
  - Tokenize văn bản và lấy trực tiếp `offset_mapping`.
  - Dự đoán chuỗi ID $\to$ Giải mã BIO trực tiếp sang span ký tự bằng hàm `bio_to_spans`.

- **Nhánh 2: Slow Tokenizer (PhoBERT)**:
  - **Tách từ (Word Seg)**: Sử dụng `viet_words_with_offsets` để lấy danh sách từ và tọa độ.
  - **Dự đoán Subword**: Đưa vào mô hình để lấy nhãn cho từng subword.
  - **Chiếu lại về Word (`subword_preds_to_word_tags`)**: Lấy nhãn của subword đầu tiên để đại diện cho cả từ.
  - **Ghép Span (`wordtags_to_spans`)**: Chuyển đổi chuỗi nhãn cấp từ về span ký tự hoàn chỉnh.

- **predict_entities(text)**:
  - API thống nhất che giấu sự phức tạp bên dưới.
  - Input: Câu văn bản thô.
  - Output: Danh sách thực thể JSON `[{start, end, label, text}, ...]`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

# Load best model/tokenizer
tok = AutoTokenizer.from_pretrained(best_dir, use_fast=True)  # nếu model là PhoBERT thì vẫn sẽ load slow tokenizer
mdl = AutoModelForTokenClassification.from_pretrained(best_dir)
mdl.eval()

with open(os.path.join(best_dir, "label_maps.json"), "r", encoding="utf-8") as f:
    maps = json.load(f)

# ensure id2label keys are int
id2label_loaded = maps["id2label"]
if isinstance(next(iter(id2label_loaded.keys())), str):
    id2label_loaded = {int(k): v for k, v in id2label_loaded.items()}

label2id_loaded = maps.get("label2id", None)
LABELS_LOADED = maps.get("labels", None)

def tokenizer_has_offsets(tokenizer) -> bool:
    """True nếu tokenizer hỗ trợ return_offsets_mapping (fast tokenizer)."""
    try:
        _ = tokenizer("test", return_offsets_mapping=True)
        return True
    except Exception:
        return False

# Helpers for PhoBERT inference (no offsets)
def subword_preds_to_word_tags(words, word_tags_gt, pred_ids, input_ids, tokenizer, id2label):
    """
    Map dự đoán subword -> word-level tag bằng cách:
    - tokenize từng word để biết số subword của word đó
    - lấy tag của subword đầu tiên làm tag word (giống cách bạn gán B/I xuống subword lúc train)
    """
    # pred_ids tương ứng với input_ids có special tokens
    # bỏ <s> ... </s>
    pred_core = pred_ids[1:-1]
    ids_core  = input_ids[1:-1]

    # Tính số subword cho mỗi word (đúng theo encode_words_and_labels)
    lens = []
    for w in words:
        ids = tokenizer.encode(w, add_special_tokens=False)
        if len(ids) == 0:
            lens.append(0)
        else:
            lens.append(len(ids))

    # Map: mỗi word lấy pred của subword đầu tiên
    word_pred_tags = []
    ptr = 0
    for L in lens:
        if L <= 0:
            # word bị skip vì encode ra rỗng -> dự đoán O
            word_pred_tags.append("O")
            continue

        if ptr >= len(pred_core):
            word_pred_tags.append("O")
            continue

        tag0 = id2label[int(pred_core[ptr])]
        word_pred_tags.append(tag0)

        ptr += L

    # Đảm bảo độ dài khớp
    if len(word_pred_tags) != len(words):
        word_pred_tags = word_pred_tags[:len(words)] + ["O"] * max(0, len(words) - len(word_pred_tags))

    return word_pred_tags


def wordtags_to_spans(text, words_info, word_tags):
    """
    words_info: [(w, ws, we), ...] offsets theo text gốc
    word_tags:  ["B-XXX"/"I-XXX"/"O", ...]
    -> list dict spans
    """
    spans = []
    cur = None  # [start,end,label]

    for (w, s, e), tag in zip(words_info, word_tags):
        if tag == "O" or tag == "IGN":
            if cur:
                spans.append(cur)
                cur = None
            continue

        pref, lab = tag.split("-", 1)

        if pref == "B" or (cur and cur[2] != lab):
            if cur:
                spans.append(cur)
            cur = [s, e, lab]
        else:
            # I
            if cur:
                cur[1] = e
            else:
                cur = [s, e, lab]

    if cur:
        spans.append(cur)

    return [{"start": s, "end": e, "label": lab, "text": text[s:e]} for s, e, lab in spans]

def predict_entities(text: str, max_length: int = 256):
    text = nfc(text)

    if tokenizer_has_offsets(tok):
        # FAST tokenizer (XLM-R)
        enc = tok(
            text,
            return_offsets_mapping=True,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
        )
        offsets = enc.pop("offset_mapping")[0].tolist()

        with torch.no_grad():
            out = mdl(**enc)

        pred_ids = out.logits.argmax(-1)[0].tolist()
        return bio_to_spans(text, offsets, pred_ids, id2label_loaded)

    else:
        # PhoBERT
        # 1) word segmentation + offsets
        words_info = viet_words_with_offsets(text)  # [(w, ws, we), ...]
        words = [w for (w, _, _) in words_info]

        # 2) encode giống train (nhãn tạm O)
        #    labels chỉ để đủ format; inference không cần labels.
        dummy_tags = ["O"] * len(words)
        enc = encode_words_and_labels(tok, words, dummy_tags, label2id_loaded, max_length=max_length)

        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]

        # 3) predict subword
        inputs = {
            "input_ids": torch.tensor([input_ids], dtype=torch.long),
            "attention_mask": torch.tensor([attention_mask], dtype=torch.long),
        }
        with torch.no_grad():
            out = mdl(**inputs)

        pred_ids = out.logits.argmax(-1)[0].tolist()

        # 4) subword -> word tag (lấy tag subword đầu tiên mỗi word)
        word_pred_tags = subword_preds_to_word_tags(
            words=words,
            word_tags_gt=dummy_tags,
            pred_ids=pred_ids,
            input_ids=input_ids,
            tokenizer=tok,
            id2label=id2label_loaded
        )

        # 5) word tags -> spans theo offsets word
        return wordtags_to_spans(text, words_info, word_pred_tags)


# Demo
print(predict_entities("Hôm nay tôi bị đau đầu và ù tai có thể đã bị si đa."))